# **Causal forest and honest splitting**

Practice for the module [“Heterogeneous effects”](https://ai-interpretability.school/learn/5/0/0), lesson “The causal forest and honest splitting”.

In the lesson, honest splitting was an argument: if the same objects choose the split and estimate the effect in a leaf, the effect estimate is biased upward, and if the roles are separated, the bias disappears and confidence intervals become available. Here we check this with numbers:

1. one tree on data where nobody has any effect: what an adaptive tree finds and what survives an honest re-estimation;
2. a causal forest with and without honest splitting: accuracy, coverage of confidence intervals, the role of the minimum leaf size;
3. the forest as a way of choosing neighbors: the weights $\alpha_i(x)$ of generalized random forests;
4. where the forest breaks: violated positivity;
5. an application to data on a growth mindset school intervention.

The notebook runs on CPU; the whole run takes about a minute.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from econml.grf import CausalForest
from econml.dml import CausalForestDML

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # spurious numpy warnings on some CPUs
SEED = 0
pd.set_option('display.width', 140)

## 1. One tree where there is no effect

A randomized experiment: 2000 objects, ten features $X\sim\mathcal N(0,I)$, a random half is treated. The outcome depends only on the first feature, $Y=x_0+\varepsilon$, and nobody has any treatment effect: $\tau(x)=0$. Any subgroup with a large effect that we find is found in noise.

We build a causal tree from an ordinary regression tree. With treatment probability $p=0.5$, the transformed outcome $Y^*=Y\,\frac{T-p}{p(1-p)}$ has conditional mean equal to the effect: $\mathbb E[Y^*\mid X=x]=\tau(x)$. A tree trained to predict $Y^*$ splits the sample where it believes the effect differs. The effect in a leaf is the difference of mean outcomes between treated and control objects.

Half A chooses the tree structure (eight leaves, at least 50 objects in each); half B is not used yet and will serve the honest re-estimation.

In [ ]:
def null_experiment(seed, n=2000, d=10):
    r = np.random.default_rng(seed)
    Xn = r.normal(size=(n, d))
    Tn = r.binomial(1, 0.5, n)
    Yn = Xn[:, 0] + r.normal(size=n)          # no effect: tau(x) = 0
    return Xn, Tn, Yn


def leaf_effects(leaves, t, y):
    # difference of mean outcomes between treated and control objects in each leaf
    ids = np.unique(leaves)
    return ids, np.array([y[(leaves == l) & (t == 1)].mean() - y[(leaves == l) & (t == 0)].mean() for l in ids])


def causal_tree(Xn, Tn, Yn, A):
    y_star = Yn * (Tn - 0.5) / 0.25            # transformed outcome: E[Y* | x] = tau(x)
    return DecisionTreeRegressor(max_leaf_nodes=8, min_samples_leaf=50, random_state=SEED).fit(Xn[A], y_star[A])


Xn, Tn, Yn = null_experiment(SEED)
A = np.arange(len(Xn)) < len(Xn) // 2          # half A: choosing the splits
B = ~A                                         # half B: honest re-estimation
tree = causal_tree(Xn, Tn, Yn, A)
ids, adaptive = leaf_effects(tree.apply(Xn[A]), Tn[A], Yn[A])
print('effect in the leaves, computed on the same half A:', np.round(adaptive, 2))

**Task 1. Re-estimate the effect in the same leaves honestly: drop the objects of half B into the leaves of `tree` and compute the difference of mean outcomes in every leaf with `leaf_effects` (array `honest`). Then repeat the whole experiment for seeds 100–199: for each seed generate data with `null_experiment`, train the tree on half A, find the leaf with the largest effect on half A, and record its effect on half A and on half B. Print the means of the two quantities over the 100 seeds.**

In [ ]:
# Your code here

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
bins = np.linspace(-1.5, 2.5, 41)
ax.hist(best_a, bins=bins, alpha=0.6, color='#c0392b', label='estimated on half A (adaptive)')
ax.hist(best_b, bins=bins, alpha=0.6, color='#3b6ea5', label='re-estimated on half B (honest)')
ax.axvline(0, color='black', lw=1, ls=':', label='true effect = 0')
ax.set_xlabel('Effect in the leaf with the largest adaptive estimate')
ax.set_ylabel('Seeds (of 100)')
ax.set_title('No effect anywhere: the "best subgroup" of one causal tree')
ax.legend(fontsize=8)
plt.show()

## 2. The forest: honest splitting, interval coverage, and leaf size

Now the effect exists and varies across objects. We take the setup from [Wager and Athey (2018)](https://arxiv.org/abs/1510.04342): five features $X\sim U[0,1]^5$, a random half treated, the effect

$$\tau(x)=\varsigma(x_0)\,\varsigma(x_1),\qquad \varsigma(u)=1+\frac{1}{1+e^{-20(u-1/3)}},$$

that is, from 1 to 4 depending on the first two features, outcome $Y=2x_2+T\tau(X)+\varepsilon$. We train `econml.grf.CausalForest` on 4000 objects and check on 2000 others where the true effect is known: the RMSE of the effect estimate, the share of objects whose true effect falls inside the 95% confidence interval (coverage), and the mean interval width.

Two hyperparameters: `honest` (honest splitting inside every tree) and `min_samples_leaf` (minimum number of objects in a leaf).

In [ ]:
def wager_athey(n, r):
    Xw = r.uniform(0, 1, (n, 5))
    Tw = r.binomial(1, 0.5, n)
    s = lambda u: 1 + 1 / (1 + np.exp(-20 * (u - 1 / 3)))
    tw = s(Xw[:, 0]) * s(Xw[:, 1])
    return Xw, Tw, 2 * Xw[:, 2] + Tw * tw + r.normal(size=n), tw


r = np.random.default_rng(1)
Xa, Ta, Ya, tau_a = wager_athey(4000, r)       # training
Xb, Tb, Yb, tau_b = wager_athey(2000, r)       # check: the true effect is needed
print(f'true effect on the check set: from {tau_b.min():.2f} to {tau_b.max():.2f}, mean {tau_b.mean():.2f}')


def forest_report(honest, min_samples_leaf):
    cf = CausalForest(n_estimators=1000, min_samples_leaf=min_samples_leaf, honest=honest, random_state=SEED).fit(Xa, Ta, Ya)
    pt, lo, hi = (a.ravel() for a in cf.predict(Xb, interval=True, alpha=0.05))
    return {'RMSE ↓': np.sqrt(np.mean((pt - tau_b) ** 2)),
            'coverage of 95% interval': np.mean((lo <= tau_b) & (tau_b <= hi)),
            'mean interval width': np.mean(hi - lo)}

**Task 2. Compute `forest_report` for `honest` in `(True, False)` and `min_samples_leaf` in `(5, 20, 50)` and build a table `grid` (rows are value pairs, columns the three quantities). Which combination has coverage closest to the nominal 95%? What happens to the coverage of the honest forest at `min_samples_leaf=50`?**

In [ ]:
# Your code here

Below-nominal coverage of the honest forest with large leaves is a consequence of bias: a large leaf gathers objects with different effects, the estimate at a point is smoothed out, and the interval built around a biased estimate misses the true value. The interval of the non-honest forest is wider and covers the true effect more often than nominal, but this is not a guarantee: the asymptotic theory of the intervals in [Wager and Athey (2018)](https://arxiv.org/abs/1510.04342) is proven only for honest trees.

## 3. The forest as a way of choosing neighbors

Generalized random forests ([Athey, Tibshirani, and Wager, 2019](https://arxiv.org/abs/1610.01271)) read a forest this way: for a point $x$, every training object $i$ gets a weight $\alpha_i(x)$, the share of trees where it lands in one leaf with $x$, normalized by the leaf size. The effect at $x$ solves a weighted estimating equation; for a binary treatment it is the weighted covariance divided by the weighted variance of the treatment:

$$\hat\tau(x)=\frac{\sum_i\alpha_i(x)(T_i-\bar T_\alpha)(Y_i-\bar Y_\alpha)}{\sum_i\alpha_i(x)(T_i-\bar T_\alpha)^2}.$$

We reconstruct the weights from the leaves of the forest (`apply`) and the subsamples of the trees (`get_subsample_inds`). Inside every tree, half of its subsample only chooses the splits; for simplicity we compute the weights over the whole subsample, so the weighted estimate will match the forest prediction approximately.

In [ ]:
cf = CausalForest(n_estimators=1000, min_samples_leaf=20, honest=True, random_state=SEED).fit(Xa, Ta, Ya)


def grf_weights(forest, Xtrain, x0):
    leaves, leaf0 = forest.apply(Xtrain), forest.apply(x0[None])[0]
    alpha = np.zeros(len(Xtrain))
    for b, sub in enumerate(forest.get_subsample_inds()):
        in_sub = np.zeros(len(Xtrain), bool)
        in_sub[sub] = True
        same = in_sub & (leaves[:, b] == leaf0[b])     # objects of tree b's subsample in the leaf of x0
        alpha[same] += 1 / same.sum()
    return alpha / len(forest.get_subsample_inds())


def weighted_effect(alpha, t, y):
    tb, yb = (alpha * t).sum(), (alpha * y).sum()
    return (alpha * (t - tb) * (y - yb)).sum() / (alpha * (t - tb) ** 2).sum()


points = np.array([[0.15, 0.15, 0.5, 0.5, 0.5], [0.8, 0.8, 0.5, 0.5, 0.5]])
s = lambda u: 1 + 1 / (1 + np.exp(-20 * (u - 1 / 3)))
alphas = [grf_weights(cf, Xa, p) for p in points]
for p, a in zip(points, alphas):
    print(f'x0={p[0]}, x1={p[1]}: true effect {s(p[0]) * s(p[1]):.2f}, forest prediction {cf.predict(p[None]).item():.2f}, '
          f'weighted estimate {weighted_effect(a, Ta, Ya):.2f}, objects with weight > 0: {(a > 0).sum()}')

Where the neighbors are. Left: the weights of the training objects in the plane of the two features the effect depends on ($x_0$, $x_1$); right: in the plane of the outcome feature $x_2$ and the noise feature $x_3$. The size and color of a dot is the weight $\alpha_i(x)$; the cross is the point $x$ itself.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8.5))
for row, (p, a) in enumerate(zip(points, alphas)):
    for ax, (i, j) in zip(axes[row], ((0, 1), (2, 3))):
        m = a > 0
        ax.scatter(Xa[~m, i], Xa[~m, j], s=2, color='#d0d0d0')
        ax.scatter(Xa[m, i], Xa[m, j], s=4 + 4000 * a[m], c=a[m], cmap='viridis', alpha=0.8)
        ax.scatter(p[i], p[j], marker='x', s=120, color='#c0392b', lw=2.5)
        ax.set_xlabel(f'x{i}')
        ax.set_ylabel(f'x{j}')
        ax.set_title(f'Point x0={p[0]}, x1={p[1]}: weights in (x{i}, x{j})')
fig.tight_layout()
plt.show()

**Task 3. For each of the two points, compute which share of the total weight $\sum_i\alpha_i(x)$ falls on training objects whose two effect features $x_0$ and $x_1$ both differ from the point's coordinates by at most 0.2. Compare with the share of such objects in the whole training sample.**

In [ ]:
# Your code here

## 4. Where the forest breaks: violated positivity

The lesson warns: where positivity is violated, the forest extrapolates and outputs a confident number. Let us check. Suppose the treatment was assigned by a rule: at $x_0<0.2$ everybody got it, at $x_0>0.8$ nobody did, in the middle a random half. The effect $\tau(x)=1+3x_0$ grows along the same feature, the outcome is $Y=2x_1+T\tau(X)+\varepsilon$. In the extreme regions there is not a single object of the opposite group, and the difference of outcomes is not observed there at all.

We train `CausalForestDML` (boosting for the outcome and propensity models) on 4000 objects and compare the RMSE, coverage, and width of intervals in three regions on 2000 check objects.

In [ ]:
r = np.random.default_rng(2)
Xp = r.uniform(0, 1, (4000, 5))
e_true = np.where(Xp[:, 0] < 0.2, 1.0, np.where(Xp[:, 0] > 0.8, 0.0, 0.5))
Tp = r.binomial(1, e_true)
Yp = 2 * Xp[:, 1] + Tp * (1 + 3 * Xp[:, 0]) + r.normal(size=4000)
Xq = r.uniform(0, 1, (2000, 5))
tau_q = 1 + 3 * Xq[:, 0]

cfp = CausalForestDML(model_y=GradientBoostingRegressor(random_state=SEED), model_t=GradientBoostingClassifier(random_state=SEED),
                      discrete_treatment=True, n_estimators=1000, min_samples_leaf=10, random_state=SEED).fit(Yp, Tp, X=Xp)
pt = cfp.effect(Xq)
lo, hi = cfp.effect_interval(Xq, alpha=0.05)
regions = {'x0 < 0.2: treated only': Xq[:, 0] < 0.2,
           '0.2 ≤ x0 ≤ 0.8: both groups': (Xq[:, 0] >= 0.2) & (Xq[:, 0] <= 0.8),
           'x0 > 0.8: control only': Xq[:, 0] > 0.8}
pos = pd.DataFrame({k: {'RMSE ↓': np.sqrt(np.mean((pt[m] - tau_q[m]) ** 2)),
                        'coverage of 95% interval': np.mean((lo[m] <= tau_q[m]) & (tau_q[m] <= hi[m])),
                        'mean interval width': np.mean(hi[m] - lo[m])} for k, m in regions.items()}).T
print(pos.round(3))

In [ ]:
order = np.argsort(Xq[:, 0])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.fill_between(Xq[order, 0], lo[order], hi[order], color='#3b6ea5', alpha=0.25, label='95% interval')
ax.plot(Xq[order, 0], pt[order], color='#3b6ea5', lw=1, label='Causal forest estimate')
ax.plot(Xq[order, 0], tau_q[order], color='#c0392b', lw=2, label='True effect 1 + 3·x0')
for xv in (0.2, 0.8):
    ax.axvline(xv, color='black', ls=':', lw=1)
ax.text(0.02, 4.3, 'treated only', fontsize=9)
ax.text(0.82, 1.2, 'control only', fontsize=9)
ax.set_xlabel('x0')
ax.set_ylabel('Effect')
ax.set_title('Causal forest without overlap: intervals barely widen where no comparison exists')
ax.legend(fontsize=8, loc='upper center')
plt.show()

**Task 4. From the table above: how many times lower is the coverage in the “treated only” region than in the region with both groups? The error of the estimate (RMSE) in the extreme regions is 4–5 times higher than in the middle, while the mean interval width is only 1.4–1.6 times larger. Why does the interval react so little to the absence of data for a comparison there?**

**Your answer:**

## 5. Application: a growth mindset program in schools

The data are a simulation based on the National Study of Learning Mindsets that the organizers of a workshop on methods for heterogeneous effects at ACIC 2018 prepared to compare methods; [Athey and Wager (2019)](https://arxiv.org/abs/1902.07409) analyzed them with a causal forest. The file `learning_mindset.csv` comes from the repository [Causal Inference for the Brave and True](https://github.com/matheusfacure/python-causality-handbook) by Matheus Facure (MIT license) and is stored in the `data/` folder of the course repository. 10391 students from 76 schools; the treatment `intervention` is a short online program teaching that abilities can grow; the outcome `achievement_score` is standardized achievement. The treatment was not assigned at random: it is related to the expectation of success `success_expect` and other features, so the outcome and propensity models are seriously needed here.

Student features are `success_expect`, `ethnicity`, `gender`, `frst_in_family`; school features are `school_urbanicity`, `school_mindset` (average growth mindset at the school), `school_achievement`, `school_ethnic_minority`, `school_poverty`, `school_size`. Categorical features are encoded with indicators. Students of one school are similar, so the cross-fitting folds are split by school (`groups`).

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/learning_mindset.csv'
lm = pd.read_csv(URL)
Xm = pd.get_dummies(lm.drop(columns=['schoolid', 'intervention', 'achievement_score']),
                    columns=['ethnicity', 'gender', 'school_urbanicity'], drop_first=True).astype(float)
Tm, Ym = lm.intervention.values, lm.achievement_score.values
print('students:', len(lm), '| schools:', lm.schoolid.nunique(), '| share treated:', round(Tm.mean(), 3))
print('naive difference of mean outcomes:', round(Ym[Tm == 1].mean() - Ym[Tm == 0].mean(), 3))

cfm = CausalForestDML(model_y=GradientBoostingRegressor(max_depth=3, random_state=SEED),
                      model_t=GradientBoostingClassifier(max_depth=3, random_state=SEED),
                      discrete_treatment=True, n_estimators=1000, min_samples_leaf=50, cv=5, random_state=SEED)
cfm.fit(Ym, Tm, X=Xm.values, groups=lm.schoolid.values)
print(cfm.ate__inference().summary_frame().round(3))
cate_m = cfm.effect(Xm.values)
print('quantiles of the CATE estimate (5%, 50%, 95%):', np.round(np.quantile(cate_m, [0.05, 0.5, 0.95]), 3))

The forest's `feature_importances_` shows which features give the splits with the largest difference in the effect between the child nodes (EconML sums this heterogeneity over the splits with a weight that decreases with depth), that is, where the forest finds differences in the effect. This is not importance for predicting the outcome: a feature that strongly affects achievement but not the effect of the program will be at the bottom here.

In [ ]:
fi = pd.Series(cfm.feature_importances_, index=Xm.columns).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(fi.index[-10:], fi.values[-10:], color='#3b6ea5')
ax.set_xlabel('Causal forest feature importance (heterogeneity from splits, depth-weighted)')
ax.set_title('Where the forest finds effect heterogeneity')
plt.tight_layout()
plt.show()

**Task 5. Split the students into quartiles of `school_mindset` (`pd.qcut(..., 4)`) and for each quartile print the mean CATE estimate and the mean 95% confidence interval of the CATE estimate (`cfm.effect_interval(Xm.values, alpha=0.05)`). Plot the means with intervals. In which quartile of school mindset does the program give a larger estimated effect?**

In [ ]:
# Your code here

The mean interval over a quartile is not an interval for the mean effect in the quartile: it describes the typical uncertainty of the CATE estimate for one student. A claim about a subgroup needs a group-level check: calibration by quantiles and the Qini curve from the notebook [“Heterogeneous effects”](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_heterogeneous_effects_en.ipynb).

## Implementations

- `econml.grf.CausalForest` is the forest itself on $(X, T, Y)$, without outcome and propensity models; suitable for randomized experiments;
- `econml.dml.CausalForestDML` is the forest on Robinson residuals, as in the R-learner; needed when treatment was not assigned at random;
- The R package [grf](https://grf-labs.github.io/grf/): `causal_forest()`, and `quantile_forest()` and `instrumental_forest()`, other forests of the same generalized random forests family.

## What to take away

- **An adaptive estimate finds an effect in noise.** Where nobody has any effect, the best leaf of a tree shows an effect of about one on the same data and about zero on the other half.
- **Honest splitting is a condition for intervals, not a decoration.** With it, coverage is close to nominal at a moderate leaf size; large leaves add bias, and coverage drops even for the honest forest.
- **A forest is weighted neighbors.** The weights $\alpha_i(x)$ concentrate on objects similar to $x$ in the effect features and barely depend on features that affect only the outcome.
- **The forest does not check positivity.** In a region without one of the groups the forest extrapolates, errs 4–5 times more, and outputs an interval only about one and a half times wider than where a comparison exists.
- **The forest's feature importance is about effect heterogeneity**, not about predicting the outcome.

Sources:
1. [Wager and Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342).
2. [Athey, Tibshirani, and Wager, 2019. Generalized random forests](https://arxiv.org/abs/1610.01271).
3. [Athey and Wager, 2019. Estimating treatment effects with causal forests: an application](https://arxiv.org/abs/1902.07409).
4. [EconML](https://github.com/py-why/EconML), [grf](https://grf-labs.github.io/grf/).